In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# 1. LOAD DATA

In [ ]:
data = load_breast_cancer()

In [ ]:
X = pd.DataFrame(data.data,columns=data.feature_names)

y = pd.Series(data.target,name="target")

print("Dataset shape:")
print(X.shape)

# 2. TRAIN / TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

# 3. CREATE LIGHTGBM MODEL

In [ ]:
model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)

# 4. TRAIN MODEL

In [ ]:
model.fit(X_train,y_train,

    eval_set=[(X_train, y_train),(X_test, y_test)],

    eval_metric="binary_logloss",
          
    callbacks=[lgb.early_stopping(stopping_rounds=50)])

# 5. PREDICTION

In [ ]:
y_pred = model.predict(X_test)

y_probability = model.predict_proba(X_test)[:, 1]

# 6. MODEL EVALUATION

In [ ]:
print("\nAccuracy:",accuracy_score(y_test,y_pred))

print("\nROC-AUC:",roc_auc_score(y_test,y_probability))

print("\nClassification Report:")
print(classification_report(y_test,y_pred,target_names=data.target_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test,y_pred))

# 7. LIGHTGBM FEATURE IMPORTANCE

In [ ]:
importance_df = pd.DataFrame({"feature":X_train.columns,
        "importance_gain":model.booster_.feature_importance(importance_type="gain"),
        "importance_split":model.booster_.feature_importance(importance_type="split")})

importance_df = importance_df.sort_values("importance_gain",ascending=False)
print("FEATURE IMPORTANCE",importance_df.head(15))

# 8. PLOT FEATURE IMPORTANCE

In [ ]:
plt.figure(figsize=(10, 7))

lgb.plot_importance(
    model,
    importance_type="gain",
    max_num_features=15)

plt.title("LightGBM Feature Importance - Gain")

plt.tight_layout()
plt.show()

# 9. LIGHTGBM PREDICTION CONTRIBUTIONS

In [ ]:
contributions = model.predict(X_test,pred_contrib=True)

print("Contribution matrix shape:",contributions.shape)

print("\nFirst observation contributions:")

print(
    pd.Series(
        contributions[0],
        index=list(X_test.columns) + ["bias"]
    ).sort_values(
        key=np.abs,
        ascending=False).head(15))